In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# আপনার বর্তমান ফোল্ডারটি খুঁজে বের করা
current_dir = os.getcwd()
env_path = os.path.join(current_dir, '.env')

# .env লোড করা
load_dotenv(dotenv_path=env_path)

api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-"):
    print("✅ API Key Found! OpenAI Client is ready.")
    client = OpenAI(api_key=api_key)
else:
    print("❌ Error: API Key পাওয়া যায়নি!")
    print(f"চেক করুন এই লোকেশনে ফাইলটি আছে কি না: {env_path}")

✅ API Key Found! OpenAI Client is ready.


In [10]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

def get_full_website_knowledge(base_url):
    all_text = ""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        # ১. হোম পেজ থেকে লিঙ্ক সংগ্রহ করা
        print(f"🌐 Connecting to: {base_url}...")
        response = requests.get(base_url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        domain = urlparse(base_url).netloc
        links = set()
        
        # দরকারি কি-ওয়ার্ড যা দেখে পেজ ফিল্টার হবে
        keywords = ['about', 'contact', 'service', 'product', 'faq', 'info', 'details']
        
        for a in soup.find_all('a', href=True):
            full_url = urljoin(base_url, a['href'])
            if domain in urlparse(full_url).netloc:
                if any(word in full_url.lower() for word in keywords):
                    links.add(full_url)
        
        # হোম পেজকেও লিস্টে রাখা
        pages_to_scrape = [base_url] + list(links)[:7]
        total_found = len(pages_to_scrape)
        
        print(f"📊 Total pages found to scrape: {total_found}")
        print("-" * 50)

        # ২. প্রতিটি পেজ থেকে ডাটা সংগ্রহ ও রিপোর্ট প্রিন্ট করা
        for index, url in enumerate(pages_to_scrape, 1):
            try:
                res = requests.get(url, headers=headers, timeout=5)
                page_soup = BeautifulSoup(res.text, 'html.parser')
                
                # অপ্রয়োজনীয় ট্যাগ পরিষ্কার করা
                for tag in page_soup(['script', 'style', 'nav', 'footer', 'header']):
                    tag.decompose()
                
                # টেক্সট প্রসেসিং
                clean_text = " ".join(page_soup.get_text(separator=' ').split())
                char_count = len(clean_text)
                
                # রিপোর্ট প্রিন্ট
                print(f"[{index}/{total_found}] 📄 Page: {url}")
                print(f"   ✨ Extracted: {char_count} characters")
                
                all_text += clean_text + " "
            except Exception as e:
                print(f"[{index}/{total_found}] ❌ Failed to scrape: {url} | Error: {e}")
                continue
                
        final_knowledge = all_text.strip()
        print("-" * 50)
        print(f"✅ Final Knowledge Base Size: {len(final_knowledge)} characters")
        return final_knowledge[:10000] # OpenAI টোকেন লিমিট মাথায় রেখে ১০k রাখা হলো
        
    except Exception as e:
        return f"Main Error: {e}"

# এখন রান করুন
target_url = "https://betopiagroup.com/" # আপনার লিঙ্কটি এখানে দিন
knowledge_base = get_full_website_knowledge(target_url)

🌐 Connecting to: https://betopiagroup.com/...
📊 Total pages found to scrape: 1
--------------------------------------------------
[1/1] 📄 Page: https://betopiagroup.com/
   ✨ Extracted: 3271 characters
--------------------------------------------------
✅ Final Knowledge Base Size: 3271 characters


In [12]:
import requests
from bs4 import BeautifulSoup

def get_master_knowledge(urls):
    master_text = ""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    print(f"🚀 Scraping started for {len(urls)} websites...")
    print("-" * 50)

    for url in urls:
        try:
            print(f"🌐 Fetching: {url}...")
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # অপ্রয়োজনীয় অংশ বাদ দেওয়া
            for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
                tag.decompose()
            
            clean_text = " ".join(soup.get_text(separator=' ').split())
            print(f"   ✅ Success! Extracted {len(clean_text)} characters.")
            
            # ডোমেইন নামসহ ডাটা সেভ করা যাতে AI বুঝতে পারে কোনটা কোন সাইটের তথ্য
            master_text += f"\n--- Source: {url} ---\n{clean_text}\n"
            
        except Exception as e:
            print(f"   ❌ Failed to fetch {url}: {e}")
            
    print("-" * 50)
    final_data = master_text.strip()
    print(f"🎯 Master Knowledge Base Ready! Total Size: {len(final_data)} characters.")
    return final_data[:15000] # OpenAI এর জন্য ১৫,০০০ ক্যারেক্টার লিমিট

# আপনার দেওয়া ওয়েবসাইট লিস্ট
target_urls = [
    "https://betopiagroup.com/",
    "https://betopialimited.com/",
    "https://fireai.agency/"
]

knowledge_base = get_master_knowledge(target_urls)

🚀 Scraping started for 3 websites...
--------------------------------------------------
🌐 Fetching: https://betopiagroup.com/...
   ✅ Success! Extracted 3271 characters.
🌐 Fetching: https://betopialimited.com/...
   ✅ Success! Extracted 9457 characters.
🌐 Fetching: https://fireai.agency/...
   ✅ Success! Extracted 5572 characters.
--------------------------------------------------
🎯 Master Knowledge Base Ready! Total Size: 18429 characters.


In [ ]:
from gtts import gTTS
import pygame
import os
import time
import speech_recognition as sr

def speak(text):
    print(f"\n🤖 AI: {text}")
    
    # অডিও ফাইল তৈরি
    tts = gTTS(text=text, lang='en')
    filename = "response.mp3"
    tts.save(filename)
    
    # pygame দিয়ে অডিও প্লে করা
    pygame.mixer.init()
    pygame.mixer.music.load(filename)
    pygame.mixer.music.play()
    
    # কথা শেষ না হওয়া পর্যন্ত অপেক্ষা করা
    while pygame.mixer.music.get_busy():
        time.sleep(0.1)
    
    # ফাইলটি রিলিজ করা যাতে পরে আবার ব্যবহার করা যায়
    pygame.mixer.quit()
    if os.path.exists(filename):
        os.remove(filename)

def run_voice_bot():
    recognizer = sr.Recognizer()
    
    with sr.Microphone() as source:
        print("\n🎤 Listening... (কথা বলুন)")
        recognizer.adjust_for_ambient_noise(source, duration=0.8)
        
        try:
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=10)
            user_query = recognizer.recognize_google(audio)
            print(f"👤 You: {user_query}")

            # OpenAI রেসপন্স
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": f"You are a helpful voice assistant for Betopia and FireAI. Data: {knowledge_base}. Keep answers very short and human-like."},
                    {"role": "user", "content": user_query}
                ]
            )
            
            reply = response.choices[0].message.content
            speak(reply) # এখানে এখন নিশ্চিতভাবে শব্দ হবে

        except sr.UnknownValueError:
            print("❓ বুঝতে পারিনি, আবার বলুন...")
        except sr.WaitTimeoutError:
            pass 
        except Exception as e:
            print(f"⚠️ Error: {e}")

# মেইন লুপ
print("\n--- 🟢 STABLE VOICE TO VOICE ACTIVE ---")
try:
    while True:
        run_voice_bot()
except KeyboardInterrupt:
    print("\n🛑 Stopped.")

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html

--- 🟢 STABLE VOICE TO VOICE ACTIVE ---

🎤 Listening... (কথা বলুন)
❓ বুঝতে পারিনি, আবার বলুন...

🎤 Listening... (কথা বলুন)
❓ বুঝতে পারিনি, আবার বলুন...

🎤 Listening... (কথা বলুন)
👤 You: can you hear me

🤖 AI: I can't hear you, but I can read the text you provide. How can I assist you today?

🎤 Listening... (কথা বলুন)
👤 You: I need a help

🤖 AI: Sure! How can I assist you today?

🎤 Listening... (কথা বলুন)
👤 You: tell me about fire AI branch

🤖 AI: Fire AI is a technology company specializing in high-performance applications, intelligent platforms, and seamless AI integrations. They focus on empowering businesses by designing intuitive digital solutions that enhance innovation and efficiency. They offer services like AI mobile app development, AI website development, AI agent development, UI/UX design, and custom software development. Fire AI is trusted by global brands and cl